# Forest Fire Detection — Image Model Benchmarking
## Notebook 03: Transfer Learning Benchmark (EfficientNet-B0, MobileNetV3, MobileNetV2, ResNet18)

**Objective**: Benchmark 4 lightweight pretrained models using short training runs to identify 
the best candidate for final training.


In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix)

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
META_DIR = IMPL / "artifacts" / "metadata"
PLOTS    = IMPL / "artifacts" / "plots"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"Torchvision: {__import__('torchvision').__version__}")

# Load split
with open(META_DIR / "data_splits.json") as f:
    splits = json.load(f)
with open(META_DIR / "class_weights.json") as f:
    cw_data = json.load(f)

CLASS_NAMES  = cw_data['class_names']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print(f"Classes: {CLASS_NAMES}")
print(f"Class to idx: {CLASS_TO_IDX}")


In [ ]:
class FireDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data  # list of [path, label]
        self.transform = transform
        self.class_to_idx = CLASS_TO_IDX

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, self.class_to_idx[label]

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Use subset for benchmarking speed
train_data_full = splits['train']
val_data_full   = splits['val']

# For benchmarking use 600 train + 200 val (fast evaluation)
import random
random.seed(SEED)
train_bench = random.sample(train_data_full, min(600, len(train_data_full)))
val_bench   = random.sample(val_data_full,   min(200, len(val_data_full)))

train_ds = FireDataset(train_bench, train_transform)
val_ds   = FireDataset(val_bench,   val_transform)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# Class weights tensor
cw = cw_data['class_weights']
weight_tensor = torch.tensor([cw['FIRE'], cw['NO_FIRE']], dtype=torch.float32).to(DEVICE)

print(f"Benchmark training set:   {len(train_bench)}")
print(f"Benchmark validation set: {len(val_bench)}")
print(f"Batch size: {BATCH_SIZE}")


In [ ]:
def build_model(model_name, num_classes=2):
    if model_name == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'MobileNetV3':
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif model_name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(DEVICE)

def freeze_backbone(model, model_name):
    """Freeze all layers except classifier head."""
    if model_name == 'EfficientNet-B0':
        for param in model.features.parameters():
            param.requires_grad = False
    elif model_name == 'MobileNetV3':
        for param in model.features.parameters():
            param.requires_grad = False
    elif model_name == 'MobileNetV2':
        for param in model.features.parameters():
            param.requires_grad = False
    elif model_name == 'ResNet18':
        for name, param in model.named_parameters():
            if 'fc' not in name:
                param.requires_grad = False

def benchmark_model(model_name, epochs=5):
    print(f"\n{'='*50}")
    print(f"  Benchmarking: {model_name}")
    print(f"{'='*50}")
    model = build_model(model_name)
    freeze_backbone(model, model_name)
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params:     {n_params:,}")
    print(f"  Trainable params: {n_trainable:,}")
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    
    train_losses, val_accs = [], []
    t_start = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        
        # Validation
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                out   = model(imgs)
                preds = out.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.cpu().numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        train_losses.append(running_loss / len(train_loader))
        val_accs.append(acc)
        print(f"  Epoch {epoch+1}/{epochs} | loss={train_losses[-1]:.4f} | val_acc={acc:.4f}")
    
    train_time = time.time() - t_start
    
    # Final metrics
    model.eval()
    all_preds, all_labels = [], []
    inf_times = []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(DEVICE)
            t0 = time.time()
            out = model(imgs)
            inf_times.append((time.time() - t0) / len(imgs) * 1000)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())
    
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    inf_ms = np.mean(inf_times)
    
    print(f"\n  Results: acc={acc:.4f} | prec={prec:.4f} | rec={rec:.4f} | f1={f1:.4f} | inf={inf_ms:.2f}ms")
    
    return {
        'model': model_name,
        'accuracy': round(acc, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1': round(f1, 4),
        'inference_ms': round(inf_ms, 2),
        'params_total': n_params,
        'params_trainable': n_trainable,
        'train_time_s': round(train_time, 1),
        'train_losses': train_losses,
        'val_accs': val_accs
    }

BENCHMARK_MODELS = ['EfficientNet-B0', 'MobileNetV3', 'MobileNetV2', 'ResNet18']
results = []
for mname in BENCHMARK_MODELS:
    r = benchmark_model(mname, epochs=5)
    results.append(r)


In [ ]:
# Display benchmark results
df_bench = pd.DataFrame([{k: v for k, v in r.items()
                           if k not in ('train_losses','val_accs')} for r in results])
df_bench = df_bench.sort_values('f1', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("BENCHMARK RESULTS — Sorted by F1 Score")
print("="*70)
print(df_bench[['model','accuracy','precision','recall','f1','inference_ms']].to_string(index=False))

# Save results
bench_dict = [{k: v for k, v in r.items() if k not in ('train_losses','val_accs')}
              for r in results]
with open(META_DIR / "benchmark_results.json", "w") as f:
    json.dump(bench_dict, f, indent=2)

best_model_name = df_bench.iloc[0]['model']
print(f"\n✓ BEST MODEL: {best_model_name}")
print(f"  F1={df_bench.iloc[0]['f1']:.4f} | Acc={df_bench.iloc[0]['accuracy']:.4f}")
with open(META_DIR / "selected_model.json", "w") as f:
    json.dump({'model_name': best_model_name}, f, indent=2)


In [ ]:
# Plot benchmark comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Model Benchmarking Results", fontsize=14, fontweight='bold')

metrics = ['accuracy', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors_models = ['#1E90FF', '#32CD32', '#FF8C00', '#DC143C']

df_plot = pd.DataFrame([{k: v for k, v in r.items() if k not in ('train_losses','val_accs')} for r in results])

for ax, metric, mlabel in zip(axes, metrics, metric_labels):
    bars = ax.bar(df_plot['model'], df_plot[metric], color=colors_models, edgecolor='black', alpha=0.85)
    ax.set_title(mlabel, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_xticklabels(df_plot['model'], rotation=20, ha='right', fontsize=8)
    for bar, val in zip(bars, df_plot[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS / "model_benchmark.png", dpi=100, bbox_inches='tight')
plt.close()
print("Benchmark plot saved.")


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for r, c in zip(results, colors_models):
    axes[0].plot(range(1, len(r['train_losses'])+1), r['train_losses'], label=r['model'], color=c, marker='o')
    axes[1].plot(range(1, len(r['val_accs'])+1), r['val_accs'], label=r['model'], color=c, marker='s')

axes[0].set_title('Training Loss (Benchmark)', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Accuracy (Benchmark)', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS / "benchmark_curves.png", dpi=100, bbox_inches='tight')
plt.close()
print(f"\nBenchmark complete. Selected model: {best_model_name}")
